# K-means Clustering — Hands-on Tutorial

In this notebook we'll build K-means from the ground up and see where it works and where it breaks:

1. The **assign-update loop** — implement one iteration by hand
2. Why **initialization** matters — same data, different answers
3. Choosing **K** with the elbow method
4. **Failure modes** — non-spherical, unequal-sized, and non-convex clusters
5. A probabilistic cousin: **Gaussian Mixture Models**

This follows the *Clustering* lecture (03_a).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_circles
from sklearn.cluster import KMeans

np.random.seed(42)

---
## Part 1: The assign-update loop

K-means repeats two steps until the assignments stop changing:

1. **Assign** each point to its nearest centroid.
2. **Update** each centroid to the mean of the points assigned to it.

Let's generate three blobs and watch it run.

In [ ]:
X, _ = make_blobs(n_samples=150, centers=3, cluster_std=0.8, random_state=42)

# Deliberately awkward starting centroids, reused by the by-hand version below
centroids = np.array([[0.0, 5.0], [0.0, -2.0], [-5.0, 0.0]])
K = 3

### Explore: watch K-means converge

Before building it by hand, get a feel for how it behaves. The widget runs K-means
on the three blobs and lets you drive it:

- **`K`** — how many centroids to look for
- **`seed`** — the random initialization (which points start as centroids)
- **`step`** — drag right to advance one assign-update iteration at a time

Watch the centroids (black ✕) march into the clusters and the inertia fall in the
title, until the run **converges** (assignments stop changing).

In [ ]:
from ipywidgets import interact, IntSlider

palette = ['#2980b9', '#e74c3c', '#27ae60', '#f39c12', '#8e44ad']

def kmeans_states(X, init, max_iter=15):
    """Return [(centroids, labels, inertia), ...], one per iteration, to convergence."""
    cents = init.astype(float).copy()
    states, prev = [], None
    for _ in range(max_iter):
        labels = np.argmin(np.linalg.norm(X[:, None] - cents[None, :], axis=2), axis=1)
        inertia = np.sum((X - cents[labels]) ** 2)
        states.append((cents.copy(), labels, inertia))
        if prev is not None and np.array_equal(labels, prev):
            break
        prev = labels
        cents = np.array([X[labels == k].mean(0) if np.any(labels == k) else cents[k]
                          for k in range(len(cents))])
    return states

def explore_kmeans(K=3, seed=0, step=0):
    rng = np.random.default_rng(seed)
    init = X[rng.choice(len(X), K, replace=False)]
    states = kmeans_states(X, init)
    step = min(step, len(states) - 1)
    cents, labels, inertia = states[step]
    plt.figure(figsize=(6, 5))
    for k in range(K):
        plt.scatter(X[labels == k, 0], X[labels == k, 1], s=15, alpha=0.6,
                    color=palette[k % len(palette)])
    plt.scatter(cents[:, 0], cents[:, 1], marker='X', s=170, color='black',
                edgecolors='white', linewidths=1.5, zorder=5)
    done = ' — CONVERGED' if step == len(states) - 1 else ''
    plt.title(f'K={K}, seed={seed} | step {step}/{len(states) - 1} | '
              f'inertia = {inertia:.0f}{done}')
    plt.xticks([]); plt.yticks([])
    plt.show()

interact(explore_kmeans,
         K=IntSlider(min=2, max=5, step=1, value=3),
         seed=IntSlider(min=0, max=9, step=1, value=0),
         step=IntSlider(min=0, max=12, step=1, value=0));

### Think about it

- Drag **`step`** rightwards. How many iterations until the title says *CONVERGED*?
  Does the inertia ever go *up* between steps?
- Keep `K=3` and try several **`seed`** values. Do they all reach the same final
  clustering? Find a seed that converges to a visibly *worse* answer — that's a
  **local minimum**, and the reason Part 2 exists.
- Set **`K=5`** on data that really has 3 blobs. Where do the two extra centroids go?

### Exercise: implement one assign-update step

Fill in the two steps below.

**Hints:**
- Distances: `np.linalg.norm(X[:, None] - centroids[None, :], axis=2)` gives an
  `(n_points, K)` matrix of distances from every point to every centroid.
- Assign: `np.argmin(..., axis=1)` picks the nearest centroid per point.
- Update: the new centroid `k` is `X[labels == k].mean(axis=0)`.

In [ ]:
def assign(X, centroids):
    dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
    return np.argmin(dists, axis=1)

def update(X, labels, K):
    return np.array([X[labels == k].mean(axis=0) for k in range(K)])

In [ ]:
# Run four iterations and plot the progression
cents = centroids.copy()
colors = ['#2980b9', '#e74c3c', '#27ae60']
fig, axes = plt.subplots(1, 4, figsize=(12, 2.8), gridspec_kw={'wspace': 0.05})
for step, ax in enumerate(axes):
    labels = assign(X, cents)
    for k in range(K):
        ax.scatter(X[labels == k, 0], X[labels == k, 1], s=12, alpha=0.6, color=colors[k])
    ax.scatter(cents[:, 0], cents[:, 1], marker='X', s=100, color='black',
               edgecolors='white', linewidths=1, zorder=5)
    ax.set_title(f'Step {step + 1}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    cents = update(X, labels, K)
plt.show()

### Think about it

- How many steps did it take before the picture stopped changing?
- K-means minimises **inertia** = total squared distance from each point to its
  centroid. Convince yourself that *both* the assign step and the update step can
  only ever decrease it. (That's why K-means always converges.)
- Convergence to *a* minimum is guaranteed — but is it the *global* one? Part 2.

---
## Part 2: Initialization matters

K-means converges to a **local** minimum of the inertia. Different starting
centroids can land you in different local minima — i.e. different clusterings of
the *same* data.

In [ ]:
X2, _ = make_blobs(n_samples=150, centers=3, cluster_std=1.2, random_state=10)

def run_kmeans(X, init, n_iter=20):
    cents = init.copy()
    for _ in range(n_iter):
        labels = assign(X, cents)
        for k in range(len(cents)):
            if np.any(labels == k):
                cents[k] = X[labels == k].mean(axis=0)
    labels = assign(X, cents)
    inertia = sum(np.sum((X[labels == k] - cents[k]) ** 2) for k in range(len(cents)))
    return labels, cents, inertia

bad_init = X2[:3]                          # three points from the same corner
good_init = X2[np.array([0, 75, 140])]     # one from each region

fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'wspace': 0.3})
for ax, init, title in [(axes[0], bad_init, 'Bad init'), (axes[1], good_init, 'Good init')]:
    labels, cents, inertia = run_kmeans(X2, init)
    for k in range(3):
        ax.scatter(X2[labels == k, 0], X2[labels == k, 1], s=15, alpha=0.6, color=colors[k])
    ax.scatter(cents[:, 0], cents[:, 1], marker='X', s=120, color='black',
               edgecolors='white', linewidths=1, zorder=5)
    ax.set_title(f'{title} (inertia = {inertia:.0f})', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### Think about it

- Which init gave the lower inertia? Is the lower-inertia clustering the one that
  matches the three blobs you can see by eye?
- scikit-learn defends against this with `n_init`: it runs K-means several times
  from different random starts and keeps the lowest-inertia result. The default
  `KMeans(n_clusters=3)` already does this — try setting `n_init=1` and re-running
  a few times to see the instability return.

---
## Part 3: Choosing K — the elbow method

In practice you don't know K. Plot inertia against K and look for the **elbow** —
the point where adding another cluster stops buying you much. (Same idea as the
scree plot for PCA.)

### Exercise: build the elbow curve

Fit `KMeans` for each K in `K_range`, collect `km.inertia_`, and plot it.

**Hints:**
- `km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X2)`
- The fitted inertia is `km.inertia_`.

In [ ]:
K_range = range(1, 9)
inertias = []
for K in K_range:
    km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X2)
    inertias.append(km.inertia_)

plt.plot(list(K_range), inertias, 'o-', color='#2980b9', lw=2, ms=8)
plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow method')
plt.grid(True, alpha=0.3)
plt.show()

### Think about it

- Where is the elbow? Does it agree with the three blobs you know are there?
- Inertia *always* decreases as K grows (more centroids → every point is closer to
  one). So why can't you just pick the K with the lowest inertia?
- The elbow is a judgement call, like the scree plot — a heuristic, not a guarantee.
  In notebook 09 you'll meet the **silhouette**; it (and the validation slides) use its peak to pick K -- unlike inertia, it actually has one.

---
## Part 4: Failure modes

K-means assumes clusters are **roughly spherical and similar-sized**, because
"nearest centroid" carves space into straight-edged (Voronoi) regions. When that
assumption is wrong, it splits or merges clusters in ways that don't match reality.

In [ ]:
from matplotlib.colors import ListedColormap

def plot_voronoi(ax, X, km, K):
    """Shade k-means' decision regions and draw the straight-line boundaries."""
    pad = 1.0
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = km.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.pcolormesh(xx, yy, Z, cmap=ListedColormap(colors[:K]), alpha=0.12,
                  shading='auto', zorder=0)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8, alpha=0.5, zorder=1)
    for k in range(K):
        ax.scatter(X[km.labels_ == k, 0], X[km.labels_ == k, 1], s=15,
                   alpha=0.7, color=colors[k], zorder=2)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], marker='X',
               s=120, color='black', edgecolors='white', linewidths=1.2, zorder=5)
    ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), gridspec_kw={'wspace': 0.3})

# 1. Non-spherical: shear the blobs into elongated streaks
X_elong, _ = make_blobs(n_samples=300, random_state=170)
X_elong = X_elong @ np.array([[0.6, -0.6], [-0.4, 0.8]])
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_elong)
plot_voronoi(axes[0], X_elong, km, 3)
axes[0].set_title('Non-spherical', fontsize=11)

# 2. Unequal sizes: one big diffuse blob, one tiny tight one
X_big = np.random.randn(300, 2) * 1.5
X_small = np.random.randn(30, 2) * 0.3 + [5, 0]
X_uneq = np.vstack([X_big, X_small])
km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_uneq)
plot_voronoi(axes[1], X_uneq, km2, 2)
axes[1].set_title('Unequal sizes', fontsize=11)

# 3. Non-convex: concentric rings
X_circ, _ = make_circles(n_samples=300, factor=0.5, noise=0.05, random_state=42)
km3 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_circ)
plot_voronoi(axes[2], X_circ, km3, 2)
axes[2].set_title('Non-convex (rings)', fontsize=11)

plt.show()

### Think about it

- The shaded tiles are k-means' **territories** — every point in a tile is assigned
  to that centroid. Notice they're always bounded by **straight lines** and are
  **convex**: that's the whole limitation in one picture.
- In the **rings** panel, no straight boundary can wrap the inner ring, so the line
  slices both rings in half. Convince yourself no choice of K fixes this.
- In the **unequal-sizes** panel, the boundary bites into the big blob to feed the
  small one — minimising total squared distance pulls it off-centre.
- Which failure is a *scaling* problem (fixable by standardising the axes) and which
  are *shape* problems the boundary can never solve? Only one is a scaling issue.

---
## Part 5: Gaussian Mixture Models

A **GMM** fits each cluster as its own Gaussian *with its own covariance*, so it can
capture elongated, tilted shapes K-means can't. It also gives **soft** assignments —
a probability of belonging to each cluster, not a hard label.

It still needs you to choose the number of components, and it's still sensitive to
initialization.

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=3, n_init=10, random_state=42).fit(X_elong)
gmm_labels = gmm.predict(X_elong)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={'wspace': 0.1})
for ax, labels, title in [(axes[0], km.labels_, 'K-means'),
                          (axes[1], gmm_labels, 'GMM')]:
    for k in range(3):
        ax.scatter(X_elong[labels == k, 0], X_elong[labels == k, 1],
                   s=15, alpha=0.6, color=colors[k])
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### Think about it

- GMM recovers the elongated streaks K-means just mangled. What did it have that
  K-means didn't? (Per-cluster covariance.)
- `gmm.predict_proba(X_elong)` gives the soft assignments. Find a point near a
  cluster boundary and look at its probabilities — they should be split, not 0/1.
- GMM still can't do the concentric rings. Why not? (Each component is still a single
  blob.) That's what **DBSCAN** is for — notebook 08.